# KBO — Application d'extraits journaliers (bronze → silver)

Pipeline incrémental : chaque dossier `KboOpenData_XXXX_Update/` contient des fichiers `{entite}_insert.csv` / `{entite}_delete.csv` pour 7 entités.
Deux régimes : **par clé** (`enterprise`, `establishment`, `branch`) et **entité entière** (`denomination`, `address`, `contact`, `activity`).
Objectif : appliquer les deltas dans l'ordre, reconstruire uniquement les entreprises affectées, sans jamais rejouer le même extrait.

## 0. Contexte

La base contient l'extrait complet **n° 431** (snapshot 24-07-2026). Trois deltas journaliers sont à appliquer dans l'ordre : **432** (25-07), **433** (26-07), **434** (27-07).
L'ensemble des entreprises affectées doit être calculé **avant** toute suppression : une fois un établissement supprimé, le lien vers son entreprise propriétaire est perdu.

## Configuration

`UPDATE_DIR = None` → traiter tous les extraits en attente dans l'ordre.
Injecter un chemin précis pour cibler un seul extrait (point d'entrée papermill/Airflow).

In [1]:
UPDATE_DIR = None        # None = tous les extraits en attente ; sinon un chemin precis
UPDATES_ROOT = None      # dossier contenant les KboOpenData_*_Update (defaut : ce dossier)

In [2]:
import csv
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pymongo
from pymongo import DeleteMany, InsertOne, ReplaceOne

sys.path.insert(0, str(Path.cwd()))
from kbo_lib import SilverTransformer, bronze_stages

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27018")
MONGO_DB = os.getenv("MONGO_DB", "kbo")

db = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)[MONGO_DB]
ROOT = Path(UPDATES_ROOT) if UPDATES_ROOT else Path.cwd()

print("mongodb :", MONGO_URI, "->", MONGO_DB)
print("extraits:", ROOT)
for name in ("entreprise", "entreprise_silver", "kbo_enterprise"):
    print(f"  {name:<20} {db[name].estimated_document_count():>12,}")
print("\nextrait de base :",
      {d["Variable"]: d["Value"] for d in db.kbo_meta.find({}, {"_id": 0})
       if d["Variable"] in ("ExtractNumber", "ExtractType", "SnapshotDate")})

mongodb : mongodb://mongo:27017 -> kbo
extraits: /home/jovyan/work
  entreprise              1,955,776
  entreprise_silver       1,955,776
  kbo_enterprise          1,955,776

extrait de base : {'SnapshotDate': '24-07-2026', 'ExtractType': 'full', 'ExtractNumber': '431'}


### Index nécessaires pour la mise à jour incrémentale

Les collections `kbo_establishment` et `kbo_branch` doivent être indexées sur leur propre clé (pas seulement sur `EnterpriseNumber`) pour retrouver et supprimer une ligne par son identifiant sans full-scan.

In [3]:
for collection, field in [("kbo_establishment", "EstablishmentNumber"),
                          ("kbo_branch", "Id"),
                          ("kbo_update_log", "appliedAt")]:
    print(f"  {collection:<20} {db[collection].create_index(field)}")

  kbo_establishment    EstablishmentNumber_1
  kbo_branch           Id_1
  kbo_update_log       appliedAt_1


---

## 1. Lecture de `meta.csv` et des fichiers insert/delete

## 1. Lecture des extraits

`Extract` encapsule un dossier d'extrait : lit `meta.csv`, expose `inserts(entite)` / `deletes(entite)`.
`ENTITIES` centralise le modèle (collection, clé, régime) — tout le pipeline en dérive.

In [4]:
# (collection bronze, cle de ligne, regime de mise a jour)
ENTITIES = {
    "enterprise":    ("kbo_enterprise",    "EnterpriseNumber",    "keyed"),
    "establishment": ("kbo_establishment", "EstablishmentNumber", "keyed"),
    "branch":        ("kbo_branch",        "Id",                  "keyed"),
    "denomination":  ("kbo_denomination",  "EntityNumber",        "whole"),
    "address":       ("kbo_address",       "EntityNumber",        "whole"),
    "contact":       ("kbo_contact",       "EntityNumber",        "whole"),
    "activity":      ("kbo_activity",      "EntityNumber",        "whole"),
}


def read_csv(path: Path) -> list[dict]:
    """Retourne les lignes d'un CSV ; liste vide si le fichier est absent."""
    if not path.exists():
        return []
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


class Extract:
    """Dossier `KboOpenData_XXXX_..._Update` : meta + accès lazy aux fichiers insert/delete."""

    def __init__(self, folder: Path):
        self.folder = Path(folder)
        rows = read_csv(self.folder / "meta.csv")
        self.meta = {row["Variable"]: row["Value"] for row in rows}
        self.number = int(self.meta["ExtractNumber"])
        self.type = self.meta["ExtractType"]
        self.snapshot = self.meta["SnapshotDate"]

    def inserts(self, entity: str) -> list[dict]:
        return read_csv(self.folder / f"{entity}_insert.csv")

    def deletes(self, entity: str) -> list[dict]:
        return read_csv(self.folder / f"{entity}_delete.csv")

    def summary(self) -> dict:
        return {entity: (len(self.inserts(entity)), len(self.deletes(entity)))
                for entity in ENTITIES}

    def __repr__(self) -> str:
        return f"<Extract {self.number} {self.type} snapshot={self.snapshot}>"


def discover(root: Path) -> list[Extract]:
    """Retourne tous les extraits du dossier, triés par numéro croissant."""
    folders = [p for p in root.iterdir() if p.is_dir() and (p / "meta.csv").exists()]
    return sorted((Extract(p) for p in folders), key=lambda e: e.number)


extracts = discover(ROOT)
print(f"{len(extracts)} extrait(s) trouve(s) :\n")
for extract in extracts:
    print(f"  {extract}")

3 extrait(s) trouve(s) :

  <Extract 432 update snapshot=25-07-2026>
  <Extract 433 update snapshot=26-07-2026>
  <Extract 434 update snapshot=27-07-2026>


In [5]:
print(f"{'entite':<16}{'inserts':>10}{'deletes':>10}   regime")
for extract in extracts:
    print(f"\n--- extrait {extract.number} (snapshot {extract.snapshot}) ---")
    for entity, (inserted, deleted) in extract.summary().items():
        regime = ENTITIES[entity][2]
        print(f"{entity:<16}{inserted:>10,}{deleted:>10,}   "
              f"{'par cle' if regime == 'keyed' else 'entite entiere'}")

entite             inserts   deletes   regime

--- extrait 432 (snapshot 25-07-2026) ---


enterprise               2         2   par cle
establishment            1         1   par cle
branch                   0         0   par cle
denomination             1         1   entite entiere
address                  1         1   entite entiere
contact                  1         1   entite entiere
activity                 1         1   entite entiere

--- extrait 433 (snapshot 26-07-2026) ---


enterprise               1         1   par cle
establishment            0         0   par cle
branch                   1         1   par cle
denomination             1         1   entite entiere
address                  0         0   entite entiere
contact                  0         0   entite entiere
activity                 1         1   entite entiere

--- extrait 434 (snapshot 27-07-2026) ---


enterprise               2         1   par cle
establishment            1         0   par cle
branch                   0         0   par cle
denomination             2         2   entite entiere
address                  1         1   entite entiere
contact                  1         1   entite entiere
activity                 1         1   entite entiere


## 2bis. Calcul de l'ensemble affecté — avant toute suppression

Trois sources : numéros d'entreprise directs, `EnterpriseNumber` porté par les inserts d'établissements/branches, et `EntityNumber` des tables de détail (préfixe `2.` → établissement, `9.` → branche, sinon entreprise directe).
Ce calcul doit précéder les deletes : une ligne supprimée ne permet plus de résoudre son entreprise propriétaire.

In [6]:
def resolve_owners(entity_numbers: set[str]) -> set[str]:
    """Résout des EntityNumber mixtes (3 niveaux) en numéros d'entreprise propriétaires.
    
    Préfixe '2.' → établissement, '9.' → branche, sinon c'est déjà une entreprise.
    """
    owners, establishments, branches = set(), [], []
    for number in entity_numbers:
        if number.startswith("2."):
            establishments.append(number)
        elif number.startswith("9."):
            branches.append(number)
        else:
            owners.add(number)

    if establishments:
        owners |= {d["EnterpriseNumber"] for d in db.kbo_establishment.find(
            {"EstablishmentNumber": {"$in": establishments}}, {"EnterpriseNumber": 1})}
    if branches:
        owners |= {d["EnterpriseNumber"] for d in db.kbo_branch.find(
            {"Id": {"$in": branches}}, {"EnterpriseNumber": 1})}
    return owners


def affected_enterprises(extract: Extract) -> set[str]:
    """Entreprises à reconstruire après cet extrait.
    
    Doit être appelé AVANT d'appliquer les suppressions : le lien entité-fille →
    entreprise disparaît dès que la ligne est supprimée du bronze.
    """
    affected = set()

    # entreprises citees directement dans les fichiers enterprise
    affected |= {r["EnterpriseNumber"] for r in extract.inserts("enterprise")}
    affected |= {r["EnterpriseNumber"] for r in extract.deletes("enterprise")}

    # etablissements et branches : insert porte EnterpriseNumber, delete necessite un lookup
    for entity, key, collection in (
            ("establishment", "EstablishmentNumber", "kbo_establishment"),
            ("branch", "Id", "kbo_branch")):
        affected |= {r["EnterpriseNumber"] for r in extract.inserts(entity)}
        keys = [r[key] for r in extract.deletes(entity)]
        if keys:
            affected |= {d["EnterpriseNumber"] for d in db[collection].find(
                {key: {"$in": keys}}, {"EnterpriseNumber": 1})}

    # tables de detail : EntityNumber melange les trois niveaux, resolution par prefixe
    entity_numbers = set()
    for entity, (_, _, regime) in ENTITIES.items():
        if regime != "whole":
            continue
        entity_numbers |= {r["EntityNumber"] for r in extract.inserts(entity)}
        entity_numbers |= {r["EntityNumber"] for r in extract.deletes(entity)}
    affected |= resolve_owners(entity_numbers)

    return affected

In [7]:
demo = extracts[0]
affected = affected_enterprises(demo)
print(f"extrait {demo.number} : {len(affected):,} entreprises affectees")
print(f"soit {len(affected) / db.kbo_enterprise.estimated_document_count():.4%} de la base")
print(f"\nechantillon : {sorted(affected)[:6]}")

orphans = len(affected) - db.kbo_enterprise.count_documents({"_id": {"$in": list(affected)}})
print(f"\ndont inconnues du bronze (creations a venir) : {orphans}")

extrait 432 : 4 entreprises affectees
soit 0.0002% de la base

echantillon : ['0200.065.765', '0200.068.636', '0200.362.210', '0403.449.823']

dont inconnues du bronze (creations a venir) : 0


## 2. Mise à jour bronze

Applique les deletes puis les inserts sur les 7 collections brutes via un seul `bulk_write` par collection.

### Idempotence du régime « entité entière »

La purge porte sur `delete ∪ insert` : sans l'union, rejouer un extrait doublerait les lignes des entités présentes uniquement dans `insert`.
Côté « par clé », `ReplaceOne(upsert=True)` offre la même garantie.

In [8]:
def apply_to_bronze(extract: Extract, *, verbose: bool = True) -> dict:
    """Applique les deletes puis les inserts sur les 7 collections brutes (un bulk_write par collection)."""
    report = {}

    for entity, (collection_name, key, regime) in ENTITIES.items():
        inserts = extract.inserts(entity)
        deletes = extract.deletes(entity)
        if not inserts and not deletes:
            report[entity] = {"deleted": 0, "inserted": 0}
            continue

        operations = []
        if regime == "keyed":
            keys = [row[key] for row in deletes]
            if keys:
                operations.append(DeleteMany({key: {"$in": keys}}))
            for row in inserts:
                document = dict(row)
                if collection_name == "kbo_enterprise":
                    document["_id"] = row[key]  # cle naturelle pour kbo_enterprise
                operations.append(ReplaceOne({key: row[key]}, document, upsert=True))
        else:
            # union delete + insert : garantit l'idempotence (pas de doublon au rejeu)
            purge = ({row["EntityNumber"] for row in deletes}
                     | {row["EntityNumber"] for row in inserts})
            if purge:
                operations.append(DeleteMany({"EntityNumber": {"$in": list(purge)}}))
            operations.extend(InsertOne(dict(row)) for row in inserts)

        result = db[collection_name].bulk_write(operations, ordered=True)
        report[entity] = {"deleted": result.deleted_count,
                          "inserted": result.inserted_count + result.upserted_count,
                          "replaced": result.modified_count}
        if verbose:
            print(f"  {entity:<16} -{result.deleted_count:>6,}  "
                  f"+{result.inserted_count + result.upserted_count:>6,}  "
                  f"~{result.modified_count:>6,}")
    return report

## 3. Propagation ciblée vers `entreprise` / `entreprise_silver`

Seules les entreprises de l'ensemble affecté sont reconstruites. Les entreprises disparues du bronze sont retirées des deux couches ; les autres passent par un `$merge` ciblé (jamais `$out`, qui remplacerait toute la collection).

### `$merge` avec `whenMatched: replace`

`$out` détruirait toute la collection ; `"merge"` conserverait les champs obsolètes. `"replace"` réécrit le document entier, ce qui est la seule option correcte pour un delta.
Sur quelques centaines d'entreprises, le pipeline `$lookup` imbriqués (trop lent sur 1,95 M) est quasi instantané : on réutilise directement `bronze_stages()`.

In [9]:
transformer = SilverTransformer.from_db(db)
print(f"{len(transformer.codes):,} codes de traduction charges")


def refresh_enterprises(numbers: set[str], *, verbose: bool = True) -> dict:
    """Reconstruit `entreprise` et `entreprise_silver` pour les entreprises données.
    
    Les entreprises disparues du bronze sont supprimées des deux couches ;
    les autres sont retraitées via $merge (jamais $out).
    """
    if not numbers:
        return {"rebuilt": 0, "removed": 0}

    target = list(numbers)
    alive = {d["_id"] for d in db.kbo_enterprise.find({"_id": {"$in": target}}, {"_id": 1})}
    removed = numbers - alive

    if removed:
        db.entreprise.delete_many({"_id": {"$in": list(removed)}})
        db.entreprise_silver.delete_many({"_id": {"$in": list(removed)}})

    if alive:
        # $merge : écrit uniquement les entreprises ciblées, sans toucher le reste
        db.kbo_enterprise.aggregate([
            {"$match": {"_id": {"$in": list(alive)}}},
            *bronze_stages(),
            {"$merge": {"into": "entreprise", "on": "_id",
                        "whenMatched": "replace", "whenNotMatched": "insert"}},
        ], allowDiskUse=True)

        operations = [
            ReplaceOne({"_id": document["_id"]}, document, upsert=True)
            for document in (transformer.to_silver(bronze)
                             for bronze in db.entreprise.find({"_id": {"$in": list(alive)}}))
        ]
        if operations:
            db.entreprise_silver.bulk_write(operations, ordered=False)

    if verbose:
        print(f"  reconstruites : {len(alive):>6,}   supprimees : {len(removed):>6,}")
    return {"rebuilt": len(alive), "removed": len(removed)}

10,641 codes de traduction charges


## 4. Journal des extraits appliqués

`kbo_update_log` enregistre chaque extrait avec `extractNumber`, `snapshotDate` et `appliedAt`. Deux gardes : `_id = extractNumber` bloque le rejeu, et on exige `numéro == dernier + 1` pour interdire un delta hors séquence.

Le point de départ de la séquence est l'extrait complet **431** lu dans `kbo_meta`. Sauter un extrait ne produit aucune erreur visible — il fausse silencieusement la base, d'où le contrôle strict `n == dernier + 1`.

In [10]:
def last_applied() -> int:
    """Numéro du dernier extrait journalier appliqué, ou numéro de l'extrait full de départ."""
    latest = db.kbo_update_log.find_one(sort=[("_id", -1)])
    if latest:
        return latest["_id"]
    base = db.kbo_meta.find_one({"Variable": "ExtractNumber"})
    return int(base["Value"]) if base else 0


def already_applied(extract: Extract) -> bool:
    return db.kbo_update_log.find_one({"_id": extract.number}) is not None


def log_extract(extract: Extract, report: dict) -> None:
    db.kbo_update_log.replace_one(
        {"_id": extract.number},
        {"_id": extract.number,
         "extractNumber": extract.number,
         "extractType": extract.type,
         "snapshotDate": extract.snapshot,
         "appliedAt": datetime.now(timezone.utc),
         **report},
        upsert=True)


print("dernier extrait applique :", last_applied())

dernier extrait applique : 431


### Orchestrateur `apply_extract`

Enchaîne les 4 étapes dans l'ordre (ensemble affecté → bronze → silver → journal) et refuse tout extrait déjà appliqué ou hors séquence.

In [11]:
def apply_extract(extract: Extract, *, verbose: bool = True) -> dict:
    """Applique un extrait en 4 étapes ordonnées, ou l'ignore s'il échoue les gardes."""
    if already_applied(extract):
        print(f"[{extract.number}] deja applique - ignore")
        return {"status": "skipped"}

    expected = last_applied() + 1
    if extract.number != expected:
        print(f"[{extract.number}] REFUSE : l'extrait attendu est {expected}. "
              f"Appliquer un delta hors sequence corromprait la base.")
        return {"status": "out-of-sequence", "expected": expected}

    started = time.perf_counter()
    print(f"[{extract.number}] snapshot {extract.snapshot}")

    # Étape 1 : calculer AVANT toute suppression (lien entité-fille perdu ensuite)
    affected = affected_enterprises(extract)
    print(f"  {len(affected):,} entreprises affectees")

    bronze_report = apply_to_bronze(extract, verbose=verbose)
    refresh_report = refresh_enterprises(affected, verbose=verbose)

    report = {"affected": len(affected),
              "rebuilt": refresh_report["rebuilt"],
              "removed": refresh_report["removed"],
              "durationSeconds": round(time.perf_counter() - started, 1),
              "bronze": bronze_report}
    log_extract(extract, report)  # écrit après succès complet (at-least-once)

    print(f"  applique en {report['durationSeconds']}s")
    return {"status": "applied", **report}

## 5. Application des extraits

Boucle sur les extraits découverts (ou sur `UPDATE_DIR` s'il est défini), triés par numéro croissant.

In [12]:
selected = [Extract(Path(UPDATE_DIR))] if UPDATE_DIR else extracts

results = []
for extract in selected:
    results.append(apply_extract(extract))
    print()

print("=" * 58)
for extract, result in zip(selected, results):
    print(f"  extrait {extract.number} : {result['status']:<16} "
          f"{result.get('affected', ''):>7} affectees  "
          f"{result.get('durationSeconds', '')!s:>6}s")

[432] snapshot 25-07-2026


  4 entreprises affectees


  enterprise       -     2  +     2  ~     0
  establishment    -     1  +     1  ~     0
  denomination     -     2  +     1  ~     0
  address          -     1  +     1  ~     0
  contact          -     1  +     1  ~     0
  activity         -     7  +     1  ~     0


  reconstruites :      4   supprimees :      0
  applique en 2.3s

[433] snapshot 26-07-2026
  2 entreprises affectees


  enterprise       -     1  +     1  ~     0
  branch           -     1  +     1  ~     0
  denomination     -     1  +     1  ~     0
  activity         -     0  +     1  ~     0
  reconstruites :      2   supprimees :      0
  applique en 1.2s

[434] snapshot 27-07-2026
  2 entreprises affectees


  enterprise       -     1  +     2  ~     0
  establishment    -     0  +     1  ~     0
  denomination     -     2  +     2  ~     0
  address          -     0  +     1  ~     0
  contact          -     0  +     1  ~     0
  activity         -     0  +     1  ~     0
  reconstruites :      2   supprimees :      0
  applique en 1.7s

  extrait 432 : applied                4 affectees     2.3s
  extrait 433 : applied                2 affectees     1.2s
  extrait 434 : applied                2 affectees     1.7s


### Historique des applications

In [13]:
print(f"{'extrait':>8} {'snapshot':>12} {'affectees':>10} {'reconstr.':>10} "
      f"{'suppr.':>8} {'duree':>7}  applique le")
for entry in db.kbo_update_log.find().sort("_id", 1):
    print(f"{entry['_id']:>8} {entry['snapshotDate']:>12} {entry['affected']:>10,} "
          f"{entry['rebuilt']:>10,} {entry['removed']:>8,} "
          f"{entry['durationSeconds']:>6}s  "
          f"{entry['appliedAt']:%Y-%m-%d %H:%M:%S}")

 extrait     snapshot  affectees  reconstr.   suppr.   duree  applique le
     432   25-07-2026          4          4        0    2.3s  2026-07-30 16:35:50
     433   26-07-2026          2          2        0    1.2s  2026-07-30 16:35:51
     434   27-07-2026          2          2        0    1.7s  2026-07-30 16:35:53


### Test d'idempotence

Rejouer les mêmes extraits : chacun doit être refusé par le journal, aucun document ne doit changer.

In [14]:
before = {name: db[name].estimated_document_count()
          for name in ("kbo_enterprise", "kbo_activity", "entreprise", "entreprise_silver")}

for extract in selected:
    apply_extract(extract)

after = {name: db[name].estimated_document_count() for name in before}
print(f"\n{'collection':<22}{'avant':>14}{'apres':>14}   ecart")
for name in before:
    print(f"{name:<22}{before[name]:>14,}{after[name]:>14,}   {after[name] - before[name]:>+,}")

[432] deja applique - ignore
[433] deja applique - ignore
[434] deja applique - ignore

collection                     avant         apres   ecart
kbo_enterprise             1,955,777     1,955,777   +0
kbo_activity              34,373,614    34,373,614   +0
entreprise                 1,955,777     1,955,777   +0
entreprise_silver          1,955,777     1,955,777   +0


## 6. Vérification

Trois contrôles : alignement des volumétries entre les trois couches, absence de documents fantômes pour les entreprises supprimées, et traçabilité bout-en-bout d'une entreprise modifiée (CSV → bronze → silver).

In [15]:
counts = {name: db[name].count_documents({}) for name in
          ("kbo_enterprise", "entreprise", "entreprise_silver")}
print("volumetrie des trois couches :")
for name, count in counts.items():
    print(f"  {name:<20} {count:>12,}")
aligned = len(set(counts.values())) == 1
print(f"  -> {'ALIGNEES' if aligned else 'ECART DETECTE'}")

# Recherche de fantomes uniquement sur le perimetre touche par les deletes enterprise.
# Un $nin sur 1,95 M de valeurs serait absurde ; on se limite aux entreprises affectees.
touched = set()
for extract in selected:
    touched |= {r["EnterpriseNumber"] for r in extract.deletes("enterprise")}

if touched:
    alive = {d["_id"] for d in db.kbo_enterprise.find({"_id": {"$in": list(touched)}}, {"_id": 1})}
    deleted = touched - alive
    ghosts_bronze = db.entreprise.count_documents({"_id": {"$in": list(deleted)}})
    ghosts_silver = db.entreprise_silver.count_documents({"_id": {"$in": list(deleted)}})
    print(f"\n{len(deleted):,} entreprises supprimees par les extraits")
    print(f"  restees dans `entreprise`        : {ghosts_bronze}   (attendu 0)")
    print(f"  restees dans `entreprise_silver` : {ghosts_silver}   (attendu 0)")

volumetrie des trois couches :
  kbo_enterprise          1,955,777
  entreprise              1,955,777
  entreprise_silver       1,955,777
  -> ALIGNEES

0 entreprises supprimees par les extraits
  restees dans `entreprise`        : 0   (attendu 0)
  restees dans `entreprise_silver` : 0   (attendu 0)


In [16]:
# Une entreprise reellement modifiee par le dernier extrait : on la suit
# depuis le CSV jusqu'au document silver.
last_extract = selected[-1]
sample_row = last_extract.inserts("enterprise")[0]
number = sample_row["EnterpriseNumber"]

print(f"entreprise {number}\n")
print("1. ligne du CSV d'insert :")
print("  ", sample_row)

print("\n2. document brut (kbo_enterprise) :")
print("  ", db.kbo_enterprise.find_one({"_id": number}))

silver = db.entreprise_silver.find_one({"_id": number})
print("\n3. document silver (traduit) :")
for key in ("status", "juridicalSituation", "juridicalForm", "typeOfEnterprise"):
    print(f"   {key:<20} {silver.get(key)}")
print(f"   denominations        {list(silver.get('denominations', {}))}")
print(f"   etablissements       {len(silver.get('establishments', {}))}")

entreprise 0200.245.711

1. ligne du CSV d'insert :
   {'EnterpriseNumber': '0200.245.711', 'Status': 'AC', 'JuridicalSituation': '012', 'TypeOfEnterprise': '2', 'JuridicalForm': '610', 'JuridicalFormCAC': '', 'StartDate': '15-03-1985'}

2. document brut (kbo_enterprise) :
   {'_id': '0200.245.711', 'EnterpriseNumber': '0200.245.711', 'Status': 'AC', 'JuridicalSituation': '012', 'TypeOfEnterprise': '2', 'JuridicalForm': '610', 'JuridicalFormCAC': '', 'StartDate': '15-03-1985'}

3. document silver (traduit) :
   status               Actif
   juridicalSituation   Dissolution volontaire – liquidation
   juridicalForm        Société à responsabilité limitée
   typeOfEnterprise     Personne morale
   denominations        ['Dénomination']
   etablissements       0


In [17]:
# Le JuridicalSituation du CSV doit se retrouver traduit dans le silver.
expected = transformer.translate("JuridicalSituation", sample_row["JuridicalSituation"])
actual = silver.get("juridicalSituation")
print(f"CSV JuridicalSituation = {sample_row['JuridicalSituation']!r}")
print(f"  attendu apres traduction : {expected!r}")
print(f"  present dans le silver   : {actual!r}")
print(f"  -> {'COHERENT' if expected == actual else 'INCOHERENT'}")

CSV JuridicalSituation = '012'
  attendu apres traduction : 'Dissolution volontaire – liquidation'
  present dans le silver   : 'Dissolution volontaire – liquidation'
  -> COHERENT


## 7. Orchestration Airflow (optionnel)

Le DAG `kbo_update_dag.py` exécute ce notebook via **papermill** une fois par jour (`0 6 * * *`, `max_active_runs=1`).
Pipeline : `list_pending → apply_updates → report`. Toute la logique métier reste dans le notebook ; Airflow ne fait que déclencher et archiver.

In [18]:
dag_path = Path("airflow/dags/kbo_update_dag.py")
print(dag_path.read_text(encoding="utf-8") if dag_path.exists()
      else f"(DAG livre dans {dag_path})")

(DAG livre dans airflow/dags/kbo_update_dag.py)


## Bilan

3 deltas appliqués sur 1,95 M d'entreprises : seules les fiches affectées sont reconstruites, chaque opération est idempotente, l'ordre de séquence est contrôlé.
Points clés : ensemble affecté calculé **avant** les deletes, `$merge` + `whenMatched: replace`, journal écrit après succès (*at-least-once*), logique métier dans le notebook — pas dans l'orchestrateur.